# Jev student (distilbert-multilingual) — Colab free-tier run

Reproduces the tony-omen v5 retrain end-to-end on a free Colab GPU.

Steps:
1. Runtime → Change runtime type → **T4 GPU**.
2. Run cell 1 to confirm the GPU.
3. Run cell 2 — a file picker appears; upload `train-all.jsonl`
   (from `scripts/jev/` in the chaba worktree, or
   `tony-omen:~/jev-retrain-20261002/train-all.jsonl`).
4. Run the rest — training ~5-8 min on T4, then the 32-case
   failure-set eval (embedded below) prints the benchmark.
5. Last cell downloads `jev-student-colab.zip` (520MB checkpoint,
   threshold embedded in config.custom_params.noul_threshold).

Corpus: 3,057 rows (live+mined+reviewed merges plus targeted aug
for Thai/embedded affirmations). Eval = golden 13 + live-divergence
13 + reviewed holds; production gate thr = 0.75.

In [ ]:
import torch, sys
print(sys.version)
print(torch.__version__, torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
!pip -q install transformers==4.46.3 scikit-learn


In [ ]:
from google.colab import files
import json, io
up = files.upload()                      # pick train-all.jsonl
name = next(iter(up))
rows = [json.loads(l) for l in io.BytesIO(up[name]).read().decode().splitlines() if l.strip()]
print(len(rows), 'rows, pos:', sum(1 for r in rows if r['label']))


In [ ]:
import json, os, random, numpy as np, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
random.seed(7); np.random.seed(7); torch.manual_seed(7)
MODEL = 'distilbert-base-multilingual-cased'

texts  = [r['text'].strip() for r in rows if r['text'].strip()]
labels = [int(r['label'])  for r in rows if r['text'].strip()]
idx = list(range(len(texts))); random.shuffle(idx)
cut = int(len(idx)*0.85); tr_i, va_i = idx[:cut], idx[cut:]

tok = AutoTokenizer.from_pretrained(MODEL)
def enc(ii):
    return tok([texts[i] for i in ii], truncation=True, max_length=96,
               padding=True, return_tensors='pt')
class DS(torch.utils.data.Dataset):
    def __init__(s, ii):
        s.e, s.y = enc(ii), torch.tensor([labels[i] for i in ii])
    def __len__(s): return len(s.y)
    def __getitem__(s,k): return {**{a:b[k] for a,b in s.e.items()},'labels':s.y[k]}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
w = torch.tensor([1.0, sum(1 for l in labels if not l)/max(1,sum(labels))])
class T(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        lab = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            out.logits, lab, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

args = TrainingArguments(output_dir='/tmp/tr', num_train_epochs=4,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, warmup_steps=10, logging_steps=50,
    eval_strategy='epoch', save_strategy='no', report_to=[],
    fp16=True, seed=7)
tr = T(model=model, args=args, train_dataset=DS(tr_i), eval_dataset=DS(va_i))
tr.train()


In [ ]:
model.eval()
def probs(ii):
    e = enc(ii)
    with torch.no_grad():
        p = torch.softmax(model(**{k:v.to(model.device) for k,v in e.items()}).logits,-1)
    return p[:,1].cpu().numpy()
va = probs(va_i); yv = np.array([labels[i] for i in va_i])
best_t, best_acc = .5, 0
for t in np.arange(.3,.9,.05):
    a = ((va>=t).astype(int)==yv).mean()
    if a>best_acc: best_acc,best_t = a,t
pred = (va>=best_t).astype(int)
tp=int((pred&(yv==1)).sum()); fp=int((pred&(yv==0)).sum()); fn=int((~pred.astype(bool)&(yv==1)).sum())
print(f'val: n={len(yv)} acc={best_acc:.3f} thr={best_t:.2f} '
      f'prec={tp/max(1,tp+fp):.2f} rec={tp/max(1,tp+fn):.2f}')
model.config.custom_params={'noul_threshold':float(best_t)}
model.save_pretrained('/content/jev-student-colab')
tok.save_pretrained('/content/jev-student-colab')


In [ ]:
# 32-case failure set (embedded: golden 13 + live divergences + reviewed)
EVAL = [
 {
  "text": "yes, publish it",
  "label": true
 },
 {
  "text": "Yes",
  "label": true
 },
 {
  "text": "go ahead",
  "label": true
 },
 {
  "text": "ยืนยันครับ",
  "label": true
 },
 {
  "text": "ใช่ ทำเลย",
  "label": true
 },
 {
  "text": "remember that the lab stack design is approved",
  "label": false
 },
 {
  "text": "turn off the TV ok",
  "label": false
 },
 {
  "text": "what is the weather today",
  "label": false
 },
 {
  "text": "save this to memory please",
  "label": false
 },
 {
  "text": "confirm delete the old page",
  "label": false
 },
 {
  "text": "Confirm the camera actually moved — ask it for the current view state.",
  "label": false
 },
 {
  "text": "เออถ้าไหนว่าไป",
  "label": true
 },
 {
  "text": "Yes, publish it.",
  "label": true
 },
 {
  "text": "Yes, publish the Thai version.",
  "label": true
 },
 {
  "text": "Yes, go ahead.",
  "label": true
 },
 {
  "text": "Yes, move the wall if it's busy — do the split.",
  "label": true
 },
 {
  "text": "Yes — do the split and put God's Eye View on the left pane.",
  "label": true
 },
 {
  "text": "Yes — set up the split and load God's Eye View on the left.",
  "label": true
 },
 {
  "text": "อ่ะ เล่น ทับ ไป เลย",
  "label": true
 },
 {
  "text": "รีเซ็ตแป๊บนึง รีเซ็ตไปเลย",
  "label": true
 },
 {
  "text": "อะ ตกลง ครับ",
  "label": true
 },
 {
  "text": "ตกลง ครับ",
  "label": true
 },
 {
  "text": "ปิด ไป เลย ปิด ไป เลย กล้อง วงจรปิด",
  "label": true
 },
 {
  "text": "ปิดไปเลยแล้วส่ง อ่า GEV มาแทน Thonburi TV",
  "label": true
 },
 {
  "text": "ให้ ดำเนิน การ ได้ เลย",
  "label": true
 },
 {
  "text": "ค่ะ เผยแพร่เลย ขอบคุณค่ะ",
  "label": true
 },
 {
  "text": "ตกลง",
  "label": true
 },
 {
  "text": "อ่าเริ่มได้เลยครับผม",
  "label": true
 },
 {
  "text": "ที่บ้านผมนี่แหละ เออ ของคุณเคเคแต่อยู่ที่บ้านผมนะ อัพเดทข้อมูลให้ที",
  "label": false
 },
 {
  "text": "เปิดวอลเปเปอร์",
  "label": false
 },
 {
  "text": "ส่ง Camwall ตัวต่อไปมาที่ทีวี",
  "label": false
 },
 {
  "text": "อามิ ส่ง Camp Wall ตัวต่อไปมาที่ทีวี",
  "label": false
 }
]
import time
res=[]
for r in EVAL:
    t=time.time()
    e=tok([r['text']],truncation=True,max_length=96,return_tensors='pt').to(model.device)
    with torch.no_grad():
        p=float(torch.softmax(model(**e).logits,-1)[0,1])
    res.append({**r,'p':p,'s':time.time()-t,'ok':(p>=0.75)==r['label']})
acc=lambda sel:(round(np.mean([x['ok'] for x in res if sel(x)]),3))
thai=lambda x:any('\u0e00'<=c<='\u0e7f' for c in x['text'])
print('acc',acc(lambda x:True),'thai',acc(thai),'pos',acc(lambda x:x['label']),
      'neg',acc(lambda x:not x['label']))
print('lat med_ms',round(np.median([x['s'] for x in res])*1000))
for x in res:
    if not x['ok']: print(' MISS',round(x['p'],3),x['label'],x['text'][:60])


In [ ]:
!cd /content && zip -qr jev-student-colab.zip jev-student-colab
from google.colab import files
files.download('/content/jev-student-colab.zip')
